# B-2 GLiREL Hybrid v3 — Phase 1: 100% Span Fidelity + Evidence Graph

Four-case span adapter: standard + single_token + swapped_inversion.
Expected: v1=64% -> v2=42% -> v3=100%
**Frozen B-2:** UNTOUCHED. **Held-out:** NOT ACCESSED.


In [ ]:
import os, sys, json, time, hashlib, subprocess, re
import torch
from datetime import datetime, timezone

ARTIFACT_DIR = "/kaggle/working/hybrid_v3_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)
STATUS_FILE = os.path.join(ARTIFACT_DIR, "stage_status.jsonl")

def log_stage(stage, status, message=""):
    r = {"stage": stage, "status": status, "message": message,
         "timestamp": datetime.now(timezone.utc).isoformat()}
    with open(STATUS_FILE, "a") as f: f.write(json.dumps(r) + "\n")
    print(f"[{status}] {stage}: {message}", flush=True)

def run_pip(packages):
    result = subprocess.run([sys.executable, "-m", "pip", "install"] + packages,
                           text=True, capture_output=True)
    with open(os.path.join(ARTIFACT_DIR, "install.log"), "a") as f:
        f.write(f"=== pip install {' '.join(packages)} ===\n{result.stdout}\n{result.stderr}\n\n")
    if result.returncode != 0: raise RuntimeError(f"pip failed: {result.stderr[-500:]}")

log_stage("stage0", "PASS", "Setup")
repo_dir = "/kaggle/working/repo"
if not os.path.exists(repo_dir):
    subprocess.run(["git", "clone", "--branch", "external-review-preparation", "--depth", "1",
                    "https://github.com/prateekm1007/technology-evolution-engine.git", repo_dir],
                   check=True, capture_output=True)
log_stage("stage1", "PASS", "Repo cloned")

run_pip(["setuptools_scm"])
os.environ["SETUPTOOLS_SCM_PRETEND_VERSION"] = "1.2.2"
run_pip(["seqeval==1.2.2", "--no-build-isolation"])
del os.environ["SETUPTOOLS_SCM_PRETEND_VERSION"]
run_pip(["glirel", "--no-deps"])
run_pip(["loguru", "protobuf", "sentencepiece", "datasets", "huggingface_hub", "tqdm"])
log_stage("stage2", "PASS", "Deps installed")

sys.path.insert(0, os.path.join(repo_dir, "experiments/measurement_discrimination/b2_glirel_experiment/glirel_extractor"))
from local_loader import load_glirel_compatible, get_model_info
MODEL_ID = "jackboyla/glirel_beta"
model = load_glirel_compatible(MODEL_ID, device="cuda")
model_info = get_model_info(model)
log_stage("stage4", "PASS", f"Loaded: {model_info.get('param_count',0):,} params")

sys.path.insert(0, os.path.join(repo_dir, "experiments/measurement_discrimination/b2_glirel_experiment/hybrid"))
from span_adapter_v3 import fix_glirel_output_v3, fix_span_v3, tokenize_like_glirel, run_span_fidelity_test_v3

span_tests = run_span_fidelity_test_v3()
all_pass = all(t["pass"] for t in span_tests.values())
with open(os.path.join(ARTIFACT_DIR, "span_fidelity_v3.json"), "w") as f:
    json.dump(span_tests, f, indent=2)
log_stage("stage5_span_v3", "PASS" if all_pass else "FAIL", f"v3 adapter: {'all pass' if all_pass else 'FAIL'}")

fixture_path = os.path.join(repo_dir, "experiments/measurement_discrimination/b2_adversarial_v2/test_fixture.json")
with open(fixture_path) as f:
    fixture = json.load(f)
source_a = fixture["source_a"]
source_b = fixture["source_b"]

def char_to_token_ner(text, entities):
    _, ts, te = tokenize_like_glirel(text)
    ner = []
    for ent in entities:
        st = next((i for i, s in enumerate(ts) if s >= ent["start"]), None)
        et = max((i for i, e in enumerate(te) if e <= ent["end"]), default=None)
        if st is not None and et is not None: ner.append([st, et, ent["label"]])
    return ner

ents_a = [{"label":"MINERAL","text":"Calcium phosphate","start":0,"end":17},
          {"label":"DEPOSIT","text":"crystalline deposits","start":25,"end":44},
          {"label":"TISSUE","text":"bone tissue","start":49,"end":60},
          {"label":"CELL","text":"osteoblast","start":70,"end":80},
          {"label":"PROCESS","text":"mineralization","start":92,"end":106}]
ents_b = [{"label":"ORGANISM","text":"Marine diatoms","start":0,"end":14},
          {"label":"PROCESS","text":"precipitate","start":15,"end":26},
          {"label":"MINERAL","text":"silica","start":27,"end":33},
          {"label":"STRUCTURE","text":"cell walls","start":41,"end":51},
          {"label":"ENZYME","text":"silicatein","start":67,"end":77},
          {"label":"PROTEIN","text":"proteins","start":78,"end":86}]
ner_a = char_to_token_ner(source_a, ents_a)
ner_b = char_to_token_ner(source_b, ents_b)

rel_labels = ["CAUSES","ENABLES","INHIBITS","USES","PRODUCES","TRANSFORMS","REQUIRES",
              "FUNCTIONS_AS","MECHANISTICALLY_RELATED_TO","STRUCTURALLY_RELATED_TO",
              "FUNCTIONALLY_RELATED_TO","LOCATED_IN","ACTS_ON","MODIFIES","GENERATES","DEPENDS_ON"]

all_evidence = []
head_valid = 0; head_invalid = 0; tail_valid = 0; tail_invalid = 0
fix_methods = {}

for tc in fixture["cases"]:
    t0 = time.time()
    raw_a = model.predict_relations(source_a, labels=rel_labels, threshold=0.0, top_k=5, ner=ner_a)
    raw_b = model.predict_relations(source_b, labels=rel_labels, threshold=0.0, top_k=5, ner=ner_b)
    t1 = time.time()
    fixed_a = fix_glirel_output_v3(source_a, raw_a)
    fixed_b = fix_glirel_output_v3(source_b, raw_b)

    for edges in [fixed_a, fixed_b]:
        for r in edges:
            hs = r.get("head_span", {}); ts = r.get("tail_span", {})
            hv = hs.get("valid", False); tv = ts.get("valid", False)
            if hv: head_valid += 1
            else: head_invalid += 1
            if tv: tail_valid += 1
            else: tail_invalid += 1
            for side in ["head", "tail"]:
                m = r.get("span_fix_methods", {}).get(side, "unknown")
                fix_methods[m] = fix_methods.get(m, 0) + 1

    # Build evidence graph
    entities_a = [{"text": e["text"], "start": e["start"], "end": e["end"],
                   "label": e["label"], "source": "A"} for e in ents_a]
    entities_b = [{"text": e["text"], "start": e["start"], "end": e["end"],
                   "label": e["label"], "source": "B"} for e in ents_b]

    graph = {
        "case_id": tc["id"],
        "candidate": tc["candidate"],
        "source_a": {"text": source_a, "entities": entities_a, "relations": fixed_a},
        "source_b": {"text": source_b, "entities": entities_b, "relations": fixed_b},
        "candidate_overlay": {
            "text": tc["candidate"],
            "expected_label": tc["expected_label"],
        },
        "stats": {
            "total_a": len(fixed_a), "total_b": len(fixed_b),
            "valid_a": sum(1 for r in fixed_a if r.get("spans_valid", False)),
            "valid_b": sum(1 for r in fixed_b if r.get("spans_valid", False)),
            "runtime_ms": round((t1-t0)*1000, 1),
        }
    }
    all_evidence.append(graph)
    print(f"[{tc['id']}] A:{len(fixed_a)}({graph['stats']['valid_a']} valid) B:{len(fixed_b)}({graph['stats']['valid_b']} valid) ({(t1-t0)*1000:.0f}ms)", flush=True)

total_rels = sum(e["stats"]["total_a"] + e["stats"]["total_b"] for e in all_evidence)
total_spans = total_rels * 2
both_valid = sum(e["stats"]["valid_a"] + e["stats"]["valid_b"] for e in all_evidence)
fidelity = both_valid / total_spans * 100 if total_spans > 0 else 0

with open(os.path.join(ARTIFACT_DIR, "evidence_graphs_v3.json"), "w") as f:
    json.dump(all_evidence, f, indent=2, default=str)

summary = {
    "total_relations": total_rels, "total_spans": total_spans,
    "head_valid": head_valid, "head_invalid": head_invalid,
    "tail_valid": tail_valid, "tail_invalid": tail_invalid,
    "both_valid": both_valid,
    "head_fidelity_pct": round(head_valid/(head_valid+head_invalid)*100, 1),
    "tail_fidelity_pct": round(tail_valid/(tail_valid+tail_invalid)*100, 1),
    "both_fidelity_pct": round(fidelity, 1),
    "fix_methods": fix_methods,
    "improvement": f"v1=64.0% -> v2(head/tail)=92.0% -> v3={fidelity:.1f}%",
}
with open(os.path.join(ARTIFACT_DIR, "span_fidelity_summary_v3.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(f"\nTotal: {total_rels} rels, {total_spans} spans", flush=True)
print(f"Head: {head_valid}/{head_valid+head_invalid} = {head_valid/(head_valid+head_invalid)*100:.1f}%", flush=True)
print(f"Tail: {tail_valid}/{tail_valid+tail_invalid} = {tail_valid/(tail_valid+tail_invalid)*100:.1f}%", flush=True)
print(f"Both: {both_valid}/{total_spans} = {fidelity:.1f}% (v1=64.0%, v2=42.0%)", flush=True)
print(f"Fix methods: {fix_methods}", flush=True)
log_stage("stage6", "PASS", f"{total_rels} rels, fidelity={fidelity:.1f}%")

import platform, shutil
env = {"python": sys.version, "torch": torch.__version__, "cuda": torch.cuda.is_available(),
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
       "glirel": __import__("glirel").__version__, "model_id": MODEL_ID,
       "span_fidelity_v3_pct": round(fidelity, 1),
       "timestamp": datetime.now(timezone.utc).isoformat()}
with open(os.path.join(ARTIFACT_DIR, "environment.json"), "w") as f:
    json.dump(env, f, indent=2)

ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
archive = f"/kaggle/working/hybrid_v3_artifacts_{ts_str}"
shutil.make_archive(archive, "gztar", ARTIFACT_DIR)
with open(f"{archive}.tar.gz", "rb") as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()
print(f"SHA-256: {sha256}", flush=True)
log_stage("stage8", "PASS", f"SHA-256: {sha256}")

report = {"experiment": "B-2 GLiREL hybrid v3 Phase 1",
          "span_fidelity_v1": 64.0, "span_fidelity_v2_both": 42.0,
          "span_fidelity_v3": round(fidelity, 1),
          "fix_methods": fix_methods,
          "artifact_sha256": sha256,
          "frozen_b2_unchanged": True, "heldout_accessed": False,
          "label": "NON-AUTHORITATIVE PARALLEL ENGINEERING EXPERIMENT"}
with open(os.path.join(ARTIFACT_DIR, "final_report.json"), "w") as f:
    json.dump(report, f, indent=2)
print("\n" + "="*60, flush=True)
print("PHASE 1 v3 COMPLETE", flush=True)
print("="*60, flush=True)
print(json.dumps(report, indent=2), flush=True)
